# Set up the data with DuckDB and Ibis

Before we build a SQL artificial intelligence (AI) agent, we need data and a database that the agent can query. In this notebook, we will follow one focused path:

`CSV → pandas DataFrame → persistent DuckDB table → SQL through Ibis`

We will use the San Francisco International Airport monthly passenger traffic dataset throughout the tutorial series. This notebook prepares its 13-column `air_traffic` table; it does not build the agent yet.

## Source data and local setup

The repository contains a snapshot of the dataset at [`data/Air_Traffic_Passenger_Statistics_20260905.csv`](../data/Air_Traffic_Passenger_Statistics_20260905.csv). The original dataset is published by [DataSF](https://data.sfgov.org/Transportation/Air-Traffic-Passenger-Statistics/rkru-6vcg/about_data).

This notebook uses Python 3.12 and the dependencies declared in [`docker/requirements.txt`](../docker/requirements.txt). From the repository root, use uv to create and activate a virtual environment, install the dependencies, and start Jupyter:

```bash
export DUCKDB_DATABASES_PATH="$HOME/databases/duckdb"
mkdir -p "$DUCKDB_DATABASES_PATH"
uv venv --python 3.12
source .venv/bin/activate
uv pip install --no-cache-dir -r docker/requirements.txt
jupyter notebook tutorials/03a_duckdb_settings.ipynb
```

DuckDB runs inside the Python process, so no separate database server is required. The database is stored as `sql-ai-agent.duckdb` under `DUCKDB_DATABASES_PATH`, which keeps the table available after the notebook session ends.

## Load the dataset

We need `os` to read the database-path environment variable, `pathlib` to work with file paths, pandas to load and prepare the CSV, and Ibis to work with DuckDB.

In [1]:
import os

from pathlib import Path

import ibis
import pandas as pd

The notebook can be launched from either the repository root or the `tutorials` directory. The next cell checks both locations and raises a clear error if the CSV is missing.

The value `NA` is a real airline code in this dataset. By disabling pandas' default missing-value list, we preserve that code while still treating empty fields as missing values.

In [2]:
filename = "Air_Traffic_Passenger_Statistics_20260905.csv"
candidates = [Path("data") / filename, Path("../data") / filename]
dataset_path = next((path.resolve() for path in candidates if path.is_file()), None)

if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find {filename}. Start Jupyter from the sql-ai-agent "
        "repository root or its tutorials directory."
    )

raw = pd.read_csv(
    dataset_path,
    keep_default_na=False,
    na_values=[""],
)
print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]} columns")

Loaded 40,450 rows and 15 columns


Defining the expected table dimensions:

In [3]:
rows_number = 40450
cols_number = 15
cols_reformat_number = 13

Let's inspect a few rows and confirm that we loaded the expected snapshot. The second assertion verifies that the literal `NA` code was not converted to a missing value.

In [4]:
raw.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,"31,432",2026/08/20 01:00:20 PM,2026/08/22 03:04:26 PM
1,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,"31,353",2026/08/20 01:00:20 PM,2026/08/22 03:04:26 PM
2,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,"2,518",2026/08/20 01:00:20 PM,2026/08/22 03:04:26 PM
3,199907,1999/07/01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,"1,324",2026/08/20 01:00:20 PM,2026/08/22 03:04:26 PM
4,199907,1999/07/01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,"1,198",2026/08/20 01:00:20 PM,2026/08/22 03:04:26 PM


In [5]:
assert raw.shape == (rows_number, cols_number)
assert raw["Operating Airline IATA Code"].eq("NA").any()
print("Source snapshot validated")

Source snapshot validated


## Prepare the `air_traffic` table

Let's review the table's columns format with the `dtypes` method:

In [6]:
raw.dtypes

Activity Period                 int64
Activity Period Start Date     object
Operating Airline              object
Operating Airline IATA Code    object
Published Airline              object
Published Airline IATA Code    object
GEO Summary                    object
GEO Region                     object
Activity Type Code             object
Price Category Code            object
Terminal                       object
Boarding Area                  object
Passenger Count                object
data_as_of                     object
data_loaded_at                 object
dtype: object

The `Passenger Count` filed, as the name implies, represents the passengers count for a given category. It currently in a string format:

In [7]:
raw["Passenger Count"].head()

0    31,432
1    31,353
2     2,518
3     1,324
4     1,198
Name: Passenger Count, dtype: object

Let's remove the commas from the field and convert it to an integer:

In [8]:
raw["Passenger Count"] = (
    raw["Passenger Count"].str.replace(",", "").astype(int)
)

Next, the source contains an activity period, its start date, and two dataset-publication timestamps. For the tutorials, we derive `Year` and `Date`, keep the dimensions used for analysis, and omit the publication timestamps.

In [9]:
raw["Date"] = pd.to_datetime(
    raw["Activity Period Start Date"],
    format="%Y/%m/%d",
)
raw["Year"] = raw["Activity Period"].astype(str).str[:4].astype(int)

Let's remove unnecessary columns:

In [10]:
table_columns = [
    "Year",
    "Date",
    "Operating Airline",
    "Operating Airline IATA Code",
    "Published Airline",
    "Published Airline IATA Code",
    "GEO Summary",
    "GEO Region",
    "Activity Type Code",
    "Price Category Code",
    "Terminal",
    "Boarding Area",
    "Passenger Count",
]

air_traffic = raw.loc[:, table_columns].copy()
air_traffic.head()


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198


A short validation confirms the row count, ordered 13-column contract, and relationship between `Year` and `Date`.

In [11]:
assert air_traffic.shape == (rows_number, cols_reformat_number)
assert air_traffic.columns.tolist() == table_columns
assert air_traffic["Year"].eq(air_traffic["Date"].dt.year).all()
print(f"Prepared table: {air_traffic.shape[0]:,} rows × {air_traffic.shape[1]} columns")

Prepared table: 40,450 rows × 13 columns


## Create a persistent DuckDB table

Ibis provides the Python connection to DuckDB and lets us execute SQL against it. We read `DUCKDB_DATABASES_PATH`, create the directory if needed, and connect to the `sql-ai-agent.duckdb` file inside it. We can then create the `air_traffic` table directly from the pandas DataFrame. Because the connection uses a file rather than an in-memory database, the table remains available after the notebook session ends.

In [12]:
duckdb_databases_path = Path(os.environ["DUCKDB_DATABASES_PATH"]).expanduser()
duckdb_databases_path.mkdir(parents=True, exist_ok=True)
database_path = duckdb_databases_path / "sql-ai-agent.duckdb"

connection = ibis.duckdb.connect(database_path)
connection.create_table("air_traffic", air_traffic, overwrite=True)

DatabaseTable: sql-ai-agent.main.air_traffic
  Year                        int64
  Date                        timestamp(6)
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

The connection now exposes the table through Ibis. We can list its tables, inspect the schema with DuckDB SQL, and run the SQL query added below to return ten rows.

In [13]:
user_tables = [
    name
    for name in connection.list_tables()
    if not name.startswith("ibis_pandas_memtable_")
]
assert "air_traffic" in user_tables
print(user_tables)
schema_sql = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'air_traffic'
ORDER BY ordinal_position
"""
connection.sql(schema_sql).execute()

['air_traffic']


,column_name,data_type
0,Year,BIGINT
1,Date,TIMESTAMP
2,Operating Airline,VARCHAR
3,Operating Airline IATA Code,VARCHAR
4,Published Airline,VARCHAR
5,Published Airline IATA Code,VARCHAR
6,GEO Summary,VARCHAR
7,GEO Region,VARCHAR
8,Activity Type Code,VARCHAR
9,Price Category Code,VARCHAR


In [14]:
connection.sql("SELECT * FROM air_traffic LIMIT 10").execute()

,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205


We can use another SQL statement to confirm that DuckDB contains the same number of rows as the prepared DataFrame.

In [15]:
row_count_result = connection.sql(
    "SELECT COUNT(*) AS row_count FROM air_traffic"
).execute()
duckdb_row_count = int(row_count_result.loc[0, "row_count"])
assert duckdb_row_count == len(air_traffic)
print(f"DuckDB row count: {duckdb_row_count:,}")

DuckDB row count: 40,450


## Query DuckDB with SQL through Ibis

The Ibis connection can execute regular SQL. Here we group the table by year and geographic summary, calculate total passengers, sort the totals, and ask DuckDB to return the first ten rows. Column names that contain spaces are quoted with double quotes.

In [16]:
passengers_by_year_sql = """
SELECT
    Year,
    "GEO Summary",
    SUM("Passenger Count") AS passenger_count
FROM air_traffic
GROUP BY Year, "GEO Summary"
ORDER BY passenger_count DESC
LIMIT 10
"""

result = connection.sql(passengers_by_year_sql).execute()
assert not result.empty
result

,Year,GEO Summary,passenger_count
0,2018,Domestic,43489998
1,2017,Domestic,42397717
2,2019,Domestic,42116019
3,2016,Domestic,40736875
4,2015,Domestic,38814323
5,2025,Domestic,38457662
6,2014,Domestic,36844841
7,2024,Domestic,36387149
8,2023,Domestic,36009025
9,2013,Domestic,35201472


## Checkpoint

You now have a prepared `air_traffic` table and have queried it with SQL through an Ibis connection. The table is stored in the `sql-ai-agent.duckdb` file under `DUCKDB_DATABASES_PATH`, so it remains available after the kernel stops. We can reconnect to the same file in a later session without loading the CSV again.

The next tutorial, **Build a SQL AI agent from scratch**, will use this same table to build the smallest complete agent. Query safety and independently enforced read-only database access will be added later in the series.

---

This tutorial is licensed under a [Creative Commons
Attribution-NonCommercial-ShareAlike 4.0
International](https://creativecommons.org/licenses/by-nc-sa/4.0/)
License.